# DATA CLEANING

In [ ]:
# # Import libraries
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np
# import statsmodels.api as sm
# from statsmodels.stats.outliers_influence import variance_inflation_factor
# from sklearn.linear_model import LinearRegression
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.metrics import mean_squared_error, r2_score

### 1.1 Random Forest - Baisc Model

In [ ]:
###############################################################################
# Random Forest - Without Hyperparameter Tuning
###############################################################################
import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Start Timer
# start_time = time.time()
# Load dataset
forest_df = pd.read_csv('motor_data14-2018.csv')
# Select relevant columns and rename them for clarity
selected_columns = {
    "SEX": "sex",
    "SEATS_NUM": "seats_num",
    "CCM_TON": "ccm_ton",
    "INSR_TYPE": "insurance_type",
    "INSURED_VALUE": "insured_value",
    "PREMIUM": "premium",
    "PROD_YEAR": "production_year",
    "TYPE_VEHICLE": "vehicle_type",
    "USAGE": "usage"
}
forest_df = forest_df[list(selected_columns.keys())].rename(columns=selected_columns)
print(forest_df.info())
zero_values = (forest_df == 0).sum()
print("Zero values in each column:")
print(zero_values)
# Drop duplicates and null values for premium and production_year
forest_df.drop_duplicates(inplace=True)
forest_df.dropna(subset=["premium", "production_year"], inplace=True)
print(forest_df.info())
zero_values = (forest_df == 0).sum()
print("Zero values in each column:")
print(zero_values)
# Ensure premium is numeric and greater than 0
forest_df = forest_df[pd.to_numeric(forest_df['premium'], errors='coerce').gt(0)]
forest_df = forest_df.sample(frac=0.1, random_state=42)
y = forest_df["premium"]
print(forest_df.info())
zero_values = (forest_df == 0).sum()
print("Zero values in each column:")
print(zero_values)
# Apply one-hot encoding to categorical variables
forest_df = pd.get_dummies(forest_df, columns=["sex", "insurance_type", "vehicle_type", "usage"], drop_first=True)
# Drop only the premium column, Train-test split
X = forest_df.drop(columns=["premium"])  
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train Random Forest
rf_model = RandomForestRegressor(max_depth=None, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)
# Performance metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print results
print("\nRandom Forest Model Basic Performance (No Log Transformation):")
print("Train RMSE:", round(train_rmse, 2))
print("Test RMSE:", round(test_rmse, 2))
print("Train MAE:", round(train_mae, 2))
print("Test MAE:", round(test_mae, 2))
print("Train R²:", round(train_r2, 3))
print("Test R²:", round(test_r2, 3))

# # End Timer
# end_time = time.time()

# # Calculate runtime
# runtime_seconds = end_time - start_time
# runtime_minutes = runtime_seconds / 60
# print("\nTotal Runtime: {:.2f} seconds ({:.2f} minutes)".format(runtime_seconds, runtime_minutes))

# Feature importance
feature_importance = rf_model.feature_importances_
feature_names = X.columns

#
importance_df_basic = pd.DataFrame({"Feature": feature_names, "Importance": feature_importance})
importance_df_basic = importance_df_basic.sort_values(by="Importance", ascending=False)


In [ ]:
# Display cleaned data info
print(forest_df.info())
print(forest_df.head())

In [ ]:
###############################################################################
# Feature Importance
###############################################################################

import plotly.express as px

#
feature_labels = {
    "insured_value": "Insured Value",
    "production_year": "Production Year",
    "ccm_ton": "Engine Capacity (Ton)",
    "seats_num": "Number of Seats",
    "usage_Fare Paying Passengers": "Vehicle Usage"
}

# Top 5 features
top_5_features = importance_df_basic.head(5).copy()
top_5_features.loc[:, "Feature"] = top_5_features["Feature"].apply(lambda x: feature_labels.get(x, x))
print("\nTop 5 Most Important Features:")
print(top_5_features)

# Plot Top 5 Feature
fig = px.bar(
    top_5_features,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Top 5 Most Important Features in Random Forest Model",
    labels={"Importance": "Importance Score", "Feature": "Feature"},
    text=top_5_features["Importance"].round(2),
)

fig.update_layout(
    yaxis=dict(categoryorder="total ascending"),
    xaxis_title="Importance Score",
    yaxis_title="Feature",
    template="plotly_white",
)

fig.show()

### 1.2 Random Forest - Optimized with Hyperparameter Tunning

In [ ]:
###############################################################################
# Random Forest - With Hyperparameter Tuning
###############################################################################

import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# # Start Timer
# start_time = time.time()

# Load dataset
forest_df = pd.read_csv('motor_data14-2018.csv')
# Select relevant columns and rename them for clarity
selected_columns = {
    "SEX": "sex",
    "SEATS_NUM": "seats_num",
    "CCM_TON": "ccm_ton",
    "INSR_TYPE": "insurance_type",
    "INSURED_VALUE": "insured_value",
    "PREMIUM": "premium",
    "PROD_YEAR": "production_year",
    "TYPE_VEHICLE": "vehicle_type",
    "USAGE": "usage"
}
forest_df = forest_df[list(selected_columns.keys())].rename(columns=selected_columns)

# Drop duplicates and null values
forest_df.drop_duplicates(inplace=True)
forest_df.dropna(subset=["premium", "production_year"], inplace=True)

# Ensure premium is numeric and greater than 0
forest_df = forest_df[pd.to_numeric(forest_df['premium'], errors='coerce').gt(0)]
forest_df = forest_df.sample(frac=0.1, random_state=42)
y = forest_df["premium"]
# Apply one-hot encoding to categorical variables
forest_df = pd.get_dummies(forest_df, columns=["sex", "insurance_type", "vehicle_type", "usage"], drop_first=True)

# Define features
X = forest_df.drop(columns=["premium"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hyperparameter tuning setup
param_dist = {
    "n_estimators": [200, 300, 400],
    "max_depth": [None, 15, 20],
    "min_samples_split": [10, 20],
    "min_samples_leaf": [3, 5, 10],
    "max_features": ["sqrt", 0.3, 0.5],
    "max_samples": [0.6, 0.8, None]
}

# Initialize Random Forest
rf = RandomForestRegressor(random_state=42, verbose=0)

# Perform Randomized Search on hyperparameter
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

# Train using randomized search
random_search.fit(X_train, y_train)
# Get the best model
best_rf = random_search.best_estimator_
print("\nBest Parameters:", random_search.best_params_)

# Retrain with best parameters, increase no. of estimator
best_rf.set_params(n_estimators=700)
best_rf.fit(X_train, y_train)

# Predictions
y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)
# Performance metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

#
print("\nTuned Model Performance (No Log Transformation):")
print("Train RMSE:", round(train_rmse, 2))
print("Test RMSE:", round(test_rmse, 2))
print("Train MAE:", round(train_mae, 2))
print("Test MAE:", round(test_mae, 2))
print("Train R²:", round(train_r2, 3))
print("Test R²:", round(test_r2, 3))

# # End Timer
# end_time = time.time()

# # Calculate runtime
# runtime_seconds = end_time - start_time
# runtime_minutes = runtime_seconds / 60
# print("\nTotal Runtime: {:.2f} seconds ({:.2f} minutes)".format(runtime_seconds, runtime_minutes))


# Feature importance
feature_importance = rf_model.feature_importances_
feature_names = X.columns

#
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": feature_importance})
importance_df = importance_df.sort_values(by="Importance", ascending=False)

In [ ]:
import plotly.express as px

feature_labels = {
    "insured_value": "Insured Value",
    "production_year": "Production Year",
    "ccm_ton": "Engine Capacity (Ton)",
    "seats_num": "Number of Seats",
    "usage_Fare Paying Passengers": "Vehicle Usage"
}

# Top 5 most important features
top_5_features = importance_df.head(5).copy()
top_5_features.loc[:, "Feature"] = top_5_features["Feature"].apply(lambda x: feature_labels.get(x, x))
print("\nTop 5 Most Important Features:")
print(top_5_features)

#
fig = px.bar(
    top_5_features,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Top 5 Most Important Features in Random Forest Model",
    labels={"Importance": "Importance Score", "Feature": "Feature"},
    text=top_5_features["Importance"].round(2),
)

fig.update_layout(
    yaxis=dict(categoryorder="total ascending"),
    xaxis_title="Importance Score",
    yaxis_title="Feature",
    template="plotly_white",
)

fig.show()

In [ ]:
# Display cleaned data info
print(forest_df.info())
print(forest_df.head())